# 09 — Precompute the AI matches for the app (Google Colab)

The app's default matching, **"AI + team check"**, needs the LLM answer for each of the 48 recipe ingredients.
The LLM needs a GPU, the demo laptop does not have one, so we compute the answers **once** here and save them to
`data/ingredient_matches_llm.csv`. The app reads that file.

**Configuration:** `Qwen2.5-14B-Instruct` + verify step, threshold 0.3 — the one selected on TUNE in notebook 08.
This notebook does **not** evaluate anything (no TEST labels are used); it only produces the AI answers.

**Before running:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all* (about 15–20 minutes).
If Colab says there is no GPU available today, wait a few hours (free daily limit).

In [ ]:
# 1. GPU check (should show a Tesla T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Code and packages
!git clone -q -b llm-matcher https://github.com/laurarigau/Food-Sustainability.git
%cd Food-Sustainability
!pip install -q -r requirements.txt
!pip install -q transformers accelerate bitsandbytes

In [ ]:
# 3. Run the LLM once on the 48 recipe ingredients
!python src/precompute_matches.py --model Qwen/Qwen2.5-14B-Instruct --verify

In [ ]:
# 4. Quick look: how often does the AI agree with the team?
import pandas as pd
ai = pd.read_csv("data/ingredient_matches_llm.csv", dtype={"agribalyse_code": str})
labels = pd.read_csv("data/ingredient_labels.csv", dtype={"agribalyse_code": str}).drop_duplicates("canonical_ingredient")
m = ai.merge(labels[["canonical_ingredient", "agribalyse_code", "agribalyse_name"]],
             left_on="ingredient", right_on="canonical_ingredient", suffixes=("_ai", "_team"))
m["agrees"] = (~m.uncertain) & (m.agribalyse_code_ai == m.agribalyse_code_team)
print(f"AI agrees with the team on {m.agrees.sum()}/{len(m)} recipe ingredients; not sure on {m.uncertain.sum()}")
print(m.loc[~m.agrees, ["ingredient", "agribalyse_name_ai", "agribalyse_name_team", "uncertain"]].to_string(index=False))

In [ ]:
# 5. Download the file and put it in the data/ folder of the repo
from google.colab import files
files.download("data/ingredient_matches_llm.csv")